# [참고·초심화] 디자인 패턴 (Design Patterns) (학생용)

> 이 단원은 **참고 사항**입니다. 지금은 "디자인 패턴이란 이런 것이구나" 정도로 이해하면 충분합니다.
> 모든 예제는 Python 3.10에서 실행해 출력 결과를 확인했습니다.

**노트북 사용법**

- 위에서부터 차례로 셀을 실행하세요 (`Shift + Enter`).
- 예제 코드는 실행하기 전에 결과를 먼저 예측해 보세요.
- 실습 문제는 `# TODO` 부분을 채운 뒤, 바로 아래 **테스트 셀**을 실행해 `테스트 통과!`가 나오면 완료입니다.

## 학습 목표

- 디자인 패턴이 무엇이고 왜 생겼는지 설명할 수 있다.
- 주요 패턴 7가지의 **목적(어떤 문제를 푸는가)**을 한 문장으로 말할 수 있다.
- 각 패턴을 파이썬으로 구현한 코드를 읽고 동작을 이해할 수 있다.
- 패턴을 쓰는 것이 오히려 과한 경우를 구분하고, **파이썬다운 단순한 대안**을 알 수 있다.

## 1. 디자인 패턴이란?

**객체 지향 설계 패턴**이란 객체 지향 프로그램을 설계하면서 **반복해서 나타나는 문제**와, 경험상 검증된 **해결 구조**에 이름을 붙여 정리한 것입니다.

- 코드 그 자체가 아니라 **설계 아이디어(템플릿)**다. 그대로 복사해서 쓰는 라이브러리가 아니다.
- 패턴 이름은 개발자끼리의 **공통 언어**가 된다. "여기는 옵저버로 처리했어요" 한마디로 구조가 전달된다.
- 실제로 여러 프로그램을 설계하며 문제를 겪어 봐야 "아, 이래서 이 패턴을 쓰는구나"를 체감할 수 있다.

### 최초의 서적: GoF의 *Design Patterns* (1994)

- 원제: *Design Patterns: Elements of Reusable Object-Oriented Software*
- 저자: Erich Gamma, Richard Helm, Ralph Johnson, John Vlissides. 네 명이라서 **GoF(Gang of Four)**라고 부른다.
- 패턴 **23가지**를 세 분류로 정리했다.
- 참고: https://en.wikipedia.org/wiki/Design_Patterns

| 분류 | 관심사 | 이 교안에서 다루는 패턴 |
| --- | --- | --- |
| 생성(Creational) 패턴 | 객체를 **어떻게 만들까** | Singleton, Builder, Factory |
| 구조(Structural) 패턴 | 객체를 **어떻게 조합할까** | Adapter, Decorator |
| 행위(Behavioral) 패턴 | 객체끼리 **어떻게 협력할까** | Observer, Strategy |

## 2. 디자인 패턴, 써야 할까?

현업에는 디자인 패턴을 **적극 권장하는 조직**도 있고, **가급적 쓰지 말라는 조직**도 있습니다.

| 관점 | 내용 |
| --- | --- |
| 장점 | 검증된 구조, 공통 언어, 확장하기 쉬움 (SOLID 원칙과 잘 맞음) |
| 단점 | 클래스가 늘고 코드 흐름이 여러 파일로 흩어져 **읽기 어려워짐** |
| 흔한 실수 | 문제가 없는데 패턴부터 적용함(과잉 설계, over-engineering) |

**정리하면**

- 기본은 **단순하고 읽기 쉬운 코드**다. 단순한 코드가 유지보수와 개발에 더 유리할 때가 많다.
- 패턴은 **문제가 실제로 생겼을 때** 해결책으로 꺼내는 도구다.
- 단, 패턴을 선호하는 조직에 들어가면 **팀의 규칙대로 철저하게** 써야 한다. 코드 일관성이 더 중요하기 때문이다.
- 파이썬은 함수가 일급 객체이고 모듈, 데코레이터, `dataclass` 같은 기능이 있다. 그래서 다른 언어에서는 클래스 여러 개가 필요한 패턴이 **몇 줄로 끝나는 경우**가 많다. 각 패턴마다 "파이썬다운 대안"을 함께 소개한다.

## 3. 생성 패턴

### 3-1. Singleton (싱글턴) 패턴

**목적**: 클래스의 객체를 **프로그램 전체에서 단 하나만** 만들고, 어디서든 그 객체를 공유한다.

**쓰는 곳**: 설정(Config), 로거(Logger), DB 연결 풀 등 하나만 있어야 하는 자원

**구현 아이디어**: 객체를 만드는 특수 메서드 `__new__`를 재정의한다. 이미 만든 객체가 있으면 새로 만들지 않고 그것을 반환한다.

In [ ]:
class AppConfig:
    _instance = None

    def __new__(cls):
        if cls._instance is None:              # 처음 한 번만 객체 생성
            print("AppConfig 객체 생성")
            cls._instance = super().__new__(cls)
            cls._instance.settings = {}
        return cls._instance                   # 이후에는 같은 객체 반환

    def set(self, key, value):
        self.settings[key] = value

    def get(self, key):
        return self.settings.get(key)


c1 = AppConfig()
c2 = AppConfig()
c1.set("theme", "dark")

print(c1 is c2)
print(id(c1) == id(c2))
print(c2.get("theme"))

**설명**

- `AppConfig()`를 두 번 호출했지만 "객체 생성"은 한 번만 출력되었다. `c1`과 `c2`는 **같은 객체**다.
- `c1`에서 저장한 값을 `c2`에서 읽을 수 있다.
- `__new__`는 객체를 **만드는** 메서드, `__init__`은 만들어진 객체를 **초기화**하는 메서드다. `__init__`은 호출될 때마다 실행되므로, 초기화를 `__init__`에 두면 값이 매번 초기화될 수 있다. 그래서 이 예제는 `__new__` 안에서 한 번만 초기화했다.

**파이썬다운 대안: 모듈**

파이썬 모듈은 처음 `import`될 때 **한 번만 실행**되고, 이후에는 같은 모듈 객체를 재사용합니다. 그래서 모듈 자체가 싱글턴처럼 동작합니다.

아래 첫 번째 셀은 `myconfig.py` 모듈 파일을 만들고, 두 번째 셀은 그 모듈을 두 번 import합니다.

In [ ]:
%%writefile myconfig.py
print("myconfig 모듈 로드")
settings = {}

In [ ]:
import myconfig
import myconfig as cfg2                 # 다시 import해도 재실행되지 않음

myconfig.settings["theme"] = "dark"
print(myconfig is cfg2, cfg2.settings)

**주의**: 싱글턴은 사실상 **전역 변수**입니다. 어디서든 값을 바꿀 수 있어 버그를 추적하기 어렵고 테스트도 어려워집니다. 그래서 **안티 패턴**으로 보는 시각도 많습니다.

### 3-2. Builder (빌더) 패턴

**목적**: 선택 항목이 많은 복잡한 객체를 **단계별로 조립**해서 만든다.

**쓰는 곳**: 매개변수가 많고 그중 상당수가 선택 사항인 객체(주문, HTTP 요청, SQL 쿼리, 문서 등)

**문제 상황**: `Pizza("L", "씬", ["페퍼로니", "올리브"], True, False, None, ...)`처럼 매개변수가 많아지면 각 값의 의미를 알기 어렵고 순서를 틀리기 쉽습니다.

In [ ]:
class Pizza:
    def __init__(self, size, dough, toppings, extra_cheese):
        self.size = size
        self.dough = dough
        self.toppings = toppings
        self.extra_cheese = extra_cheese

    def __str__(self):
        cheese = " + 치즈 추가" if self.extra_cheese else ""
        return f"{self.size} / {self.dough} / {', '.join(self.toppings) or '토핑 없음'}{cheese}"


class PizzaBuilder:
    def __init__(self, size):                  # 필수 값
        self._size = size
        self._dough = "오리지널"                # 선택 값의 기본값
        self._toppings = []
        self._extra_cheese = False

    def dough(self, dough):
        self._dough = dough
        return self                            # self 반환 -> 메서드 체이닝

    def topping(self, name):
        self._toppings.append(name)
        return self

    def extra_cheese(self):
        self._extra_cheese = True
        return self

    def build(self):                           # 최종 검증 후 객체 생성
        if len(self._toppings) > 3:
            raise ValueError("토핑은 최대 3개까지 가능합니다")
        return Pizza(self._size, self._dough, list(self._toppings), self._extra_cheese)


p1 = PizzaBuilder("L").dough("씬").topping("페퍼로니").topping("올리브").extra_cheese().build()
p2 = PizzaBuilder("M").build()
print(p1)
print(p2)

try:
    b = PizzaBuilder("S")
    for t in ["햄", "버섯", "양파", "옥수수"]:
        b.topping(t)
    b.build()
except ValueError as e:
    print("ValueError:", e)

**설명**

- 각 설정 메서드가 `self`를 반환하므로 `.dough().topping().build()`처럼 **메서드 체이닝**을 할 수 있다.
- 값 검증은 `build()`에서 한 번에 한다. 그래서 완성된 `Pizza` 객체는 항상 올바른 상태다.

**파이썬다운 대안: 키워드 인자 + 기본값 (`dataclass`)**

파이썬은 **키워드 인자와 기본값**을 지원하므로, 단순한 경우에는 빌더 없이도 읽기 쉽게 만들 수 있습니다.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Pizza:
    size: str
    dough: str = "오리지널"
    toppings: list = field(default_factory=list)
    extra_cheese: bool = False

print(Pizza("L", dough="씬", toppings=["페퍼로니"], extra_cheese=True))

> 빌더는 조립 과정이 복잡하거나, 단계별 검증이 필요하거나, 같은 과정으로 다른 결과물을 만들어야 할 때 고려합니다.

### 3-3. Factory (팩토리) 패턴

**목적**: 객체 **생성 코드를 한곳에 모아**, 사용하는 쪽이 구체 클래스 이름을 몰라도 객체를 얻게 한다.

**쓰는 곳**: 설정값이나 사용자 입력에 따라 다른 클래스의 객체를 만들어야 할 때 (알림 방식, 파일 형식별 파서, DB 종류 등)

> GoF 원서에는 Factory Method, Abstract Factory 패턴이 있습니다. 여기서는 이해하기 쉬운 **단순 팩토리(Simple Factory)** 형태로 핵심 아이디어를 봅니다.

In [ ]:
from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def send(self, msg): ...

class EmailNotifier(Notifier):
    def send(self, msg):
        return f"이메일 발송: {msg}"

class SMSNotifier(Notifier):
    def send(self, msg):
        return f"문자 발송: {msg}"

class PushNotifier(Notifier):
    def send(self, msg):
        return f"앱 푸시 발송: {msg}"


class NotifierFactory:
    _registry = {
        "email": EmailNotifier,
        "sms": SMSNotifier,
        "push": PushNotifier,
    }

    @classmethod
    def create(cls, kind) -> Notifier:          # 어떤 클래스를 만들지 여기서 결정
        try:
            return cls._registry[kind]()
        except KeyError:
            raise ValueError(f"지원하지 않는 알림 방식: {kind}") from None

    @classmethod
    def register(cls, kind, notifier_cls):      # 새 방식 등록 (OCP)
        cls._registry[kind] = notifier_cls


# 사용하는 쪽은 구체 클래스 이름을 몰라도 됨 (설정값/사용자 입력 등)
for kind in ["email", "sms", "push"]:
    print(NotifierFactory.create(kind).send("주문 완료"))

class KakaoNotifier(Notifier):
    def send(self, msg):
        return f"카카오톡 발송: {msg}"

NotifierFactory.register("kakao", KakaoNotifier)
print(NotifierFactory.create("kakao").send("배송 시작"))

try:
    NotifierFactory.create("fax")
except ValueError as e:
    print("ValueError:", e)

**설명**

- 사용하는 쪽은 `"email"`, `"sms"` 같은 **문자열만** 알면 된다. `EmailNotifier`라는 클래스 이름을 몰라도 된다.
- `if/elif` 대신 **딕셔너리(레지스트리)**로 클래스를 연결했다. 새 방식은 `register()`로 추가할 수 있다(OCP).
- 파이썬에서는 클래스도 객체이므로 `cls._registry[kind]()`처럼 딕셔너리에서 꺼낸 클래스를 바로 호출해 객체를 만들 수 있다. 이 딕셔너리 방식 자체가 이미 파이썬다운 팩토리다.

## 4. 구조 패턴

### 4-1. Adapter (어댑터) 패턴

**목적**: 인터페이스가 맞지 않는 객체를 **변환기로 감싸** 우리 코드에서 쓸 수 있게 한다. 해외여행용 **전원 어댑터**를 생각하면 된다.

**쓰는 곳**: 외부 라이브러리나 레거시 코드를 수정하지 않고 우리 시스템에 붙일 때

In [ ]:
# 우리 시스템이 기대하는 인터페이스: pay(amount) -> bool
class PaymentGateway:
    def pay(self, amount: int) -> bool:
        raise NotImplementedError

class OurPay(PaymentGateway):
    def pay(self, amount):
        print(f"OurPay로 {amount:,}원 결제")
        return True


# 외부 라이브러리(수정 불가): 메서드 이름·단위·반환값이 다름
class ForeignPayAPI:
    def make_payment(self, cents: int, currency: str) -> dict:
        print(f"ForeignPay: {cents} cents ({currency}) 결제 요청")
        return {"status": "OK"}


class ForeignPayAdapter(PaymentGateway):       # 외부 객체를 우리 인터페이스에 맞춤
    RATE = 1300                                  # 1 USD = 1300원 (예시 환율)

    def __init__(self, api: ForeignPayAPI):
        self.api = api

    def pay(self, amount):
        cents = round(amount / self.RATE * 100)
        result = self.api.make_payment(cents, "USD")
        return result["status"] == "OK"


def checkout(gateway: PaymentGateway, amount):   # 클라이언트 코드는 그대로
    ok = gateway.pay(amount)
    print("결제 성공" if ok else "결제 실패")

checkout(OurPay(), 26_000)
checkout(ForeignPayAdapter(ForeignPayAPI()), 26_000)

**설명**

- `ForeignPayAPI`는 메서드 이름(`make_payment`), 금액 단위(센트), 반환값(dict)이 모두 다르다.
- `ForeignPayAdapter`가 중간에서 변환해 주므로 `checkout()` 코드는 **한 줄도 바뀌지 않았다**.
- 외부 코드도, 우리 코드도 수정하지 않는다는 점이 핵심이다.

### 4-2. Decorator (데코레이터) 패턴

**목적**: 객체를 **감싸서** 기존 코드를 수정하지 않고 기능을 덧붙인다. 여러 겹으로 감쌀 수 있다.

**쓰는 곳**: 옵션 조합이 많아 상속으로 처리하면 클래스 수가 폭발할 때 (예: 샷추가아메리카노, 시럽샷추가아메리카노, ...)

In [ ]:
from abc import ABC, abstractmethod

class Coffee(ABC):
    @abstractmethod
    def cost(self): ...
    @abstractmethod
    def desc(self): ...

class Americano(Coffee):
    def cost(self):
        return 4000
    def desc(self):
        return "아메리카노"


class CoffeeDecorator(Coffee):                 # 같은 인터페이스를 구현하면서
    def __init__(self, coffee: Coffee):         # 원래 객체를 감싼다
        self._coffee = coffee

class Shot(CoffeeDecorator):
    def cost(self):
        return self._coffee.cost() + 500
    def desc(self):
        return self._coffee.desc() + " + 샷"

class Syrup(CoffeeDecorator):
    def cost(self):
        return self._coffee.cost() + 300
    def desc(self):
        return self._coffee.desc() + " + 시럽"

class Whip(CoffeeDecorator):
    def cost(self):
        return self._coffee.cost() + 700
    def desc(self):
        return self._coffee.desc() + " + 휘핑"


menu = [
    Americano(),
    Shot(Americano()),
    Whip(Syrup(Shot(Shot(Americano())))),      # 원하는 만큼 겹쳐 감싸기
]
for c in menu:
    print(f"{c.desc()} = {c.cost():,}원")

**설명**

- `Shot`, `Syrup`, `Whip`은 모두 `Coffee`이면서 **안에 다른 `Coffee`를 품고** 있다. 그래서 몇 겹이든 감쌀 수 있다.
- 바깥 객체의 `cost()`는 안쪽 `cost()`를 호출한 뒤 자기 가격을 더한다. 이 호출이 안쪽으로 차례로 전달된다.
- 옵션이 3개면 상속으로는 조합별 클래스가 최대 2³ = 8개 필요하다. 데코레이터는 **3개**로 충분하다.

**파이썬의 `@데코레이터` 문법과의 관계**

파이썬의 `@` 데코레이터 문법은 GoF 데코레이터 패턴과 **아이디어(감싸서 기능 추가)는 같지만**, 객체가 아니라 **함수**를 감싸는 언어 기능입니다.

In [ ]:
import functools

def log_call(func):                            # 함수를 감싸 기능을 덧붙임
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[호출] {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"[결과] {result}")
        return result
    return wrapper

@log_call
def add(a, b):
    return a + b

add(2, 3)

## 5. 행위 패턴

### 5-1. Observer (옵저버) 패턴

**목적**: 한 객체(발행자, Subject)의 상태가 바뀌면, 그 객체를 **구독한 객체들(Observer)에게 자동으로 알린다**.

**쓰는 곳**: 이벤트 처리(버튼 클릭), 가격·재고 알림, 유튜브 구독 알림, GUI·게임 이벤트 시스템

In [ ]:
from abc import ABC, abstractmethod

class Observer(ABC):                         # 구독자 인터페이스
    @abstractmethod
    def update(self, product, price): ...

class Product:                               # 발행자(Subject)
    def __init__(self, name, price):
        self.name = name
        self._price = price
        self._observers = []

    def attach(self, observer):
        self._observers.append(observer)

    def detach(self, observer):
        self._observers.remove(observer)

    def _notify(self):
        for ob in self._observers:
            ob.update(self, self._price)

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):                  # 가격이 바뀌면 자동 통지
        self._price = value
        self._notify()

class EmailAlert(Observer):
    def __init__(self, email):
        self.email = email
    def update(self, product, price):
        print(f"[메일 → {self.email}] {product.name} 가격 변경: {price:,}원")

class PriceHistory(Observer):
    def __init__(self):
        self.history = []
    def update(self, product, price):
        self.history.append(price)


phone = Product("스마트폰", 1_000_000)
alert = EmailAlert("kim@example.com")
history = PriceHistory()

phone.attach(alert)
phone.attach(history)

phone.price = 950_000
phone.price = 900_000
phone.detach(alert)                          # 구독 해지
phone.price = 850_000

print("가격 기록:", history.history)

**설명**

- `Product`는 구독자가 누구인지(메일인지, 기록인지) **구체적으로 알 필요가 없다**. `update()`만 호출한다. 이런 관계를 **느슨한 결합**이라고 한다.
- `@property`의 setter를 이용해 `phone.price = ...` 대입만으로 통지가 일어나게 했다.
- 메일 구독을 해지(`detach`)한 뒤 세 번째 가격 변경은 기록에만 남았다.
- 새 알림 방식(문자, 슬랙 등)이 필요하면 `Observer`를 구현한 클래스만 추가하면 된다.

### 5-2. Strategy (전략) 패턴

**목적**: 바꿔 끼울 수 있는 **알고리즘(전략)들을 각각 객체로 분리**하고, 실행 중에 골라 쓴다.

**쓰는 곳**: 할인 정책, 정렬 기준, 결제 방식, 경로 탐색 방식 등 "방법"만 다르고 "목적"은 같은 경우

In [ ]:
from abc import ABC, abstractmethod

class DiscountStrategy(ABC):
    @abstractmethod
    def apply(self, price): ...

class NoDiscount(DiscountStrategy):
    def apply(self, price):
        return price

class PercentDiscount(DiscountStrategy):
    def __init__(self, percent):
        self.percent = percent
    def apply(self, price):
        return int(price * (1 - self.percent / 100))

class FixedDiscount(DiscountStrategy):
    def __init__(self, amount):
        self.amount = amount
    def apply(self, price):
        return max(price - self.amount, 0)


class Order:
    def __init__(self, price, strategy: DiscountStrategy):
        self.price = price
        self.strategy = strategy               # 알고리즘을 객체로 주입

    def total(self):
        return self.strategy.apply(self.price)


order = Order(50_000, NoDiscount())
print("기본    :", order.total())

order.strategy = PercentDiscount(10)           # 실행 중에 전략 교체
print("10% 할인:", order.total())

order.strategy = FixedDiscount(7_000)
print("7천원 할인:", order.total())

**설명**

- `Order`는 할인 계산 방법을 모른다. 전달받은 전략의 `apply()`를 호출할 뿐이다.
- 새 할인 정책을 추가해도 `Order`는 수정하지 않는다(OCP). 전략 객체를 주입하는 구조는 DIP와도 같다.
- **팩토리 vs 전략**: 팩토리는 "**무엇을 만들지**", 전략은 "**어떻게 처리할지**"를 바꾼다.

**파이썬다운 대안: 함수 전달**

파이썬에서는 함수도 객체이므로, 전략이 메서드 하나뿐이라면 **클래스 대신 함수**를 넘기면 됩니다. `sorted(data, key=len)`의 `key`가 바로 이 방식입니다.

In [ ]:
def no_discount(price):
    return price

def percent(p):
    return lambda price: int(price * (1 - p / 100))

def total(price, discount=no_discount):
    return discount(price)

print(total(50_000))
print(total(50_000, percent(10)))

## 6. 정리

| 패턴 | 분류 | 한 줄 요약 | 파이썬다운 대안 |
| --- | --- | --- | --- |
| Singleton | 생성 | 객체를 하나만 만들어 공유 | 모듈 |
| Builder | 생성 | 복잡한 객체를 단계별로 조립 | 키워드 인자, `dataclass` |
| Factory | 생성 | 객체 생성 코드를 한곳에 모음 | 클래스를 담은 딕셔너리 |
| Adapter | 구조 | 맞지 않는 인터페이스를 변환 | (그대로 사용) |
| Decorator | 구조 | 객체를 감싸 기능을 덧붙임 | 함수에는 `@` 데코레이터 |
| Observer | 행위 | 상태 변화를 구독자에게 자동 통지 | 콜백 함수 리스트 |
| Strategy | 행위 | 알고리즘을 객체로 분리해 교체 | 함수 전달 (`key=`) |

**기억할 것**

- 패턴은 **목적**부터 이해한다. 구조(클래스 다이어그램)는 그다음이다.
- 대부분의 패턴은 앞 단원의 **SOLID 원칙**(특히 OCP, DIP)을 구현하는 구체적인 방법이다.
- 문제가 없는데 패턴부터 쓰지 않는다. 단순한 코드가 먼저다.

## 실습 문제

각 문제의 `# TODO`를 채우고, 아래 테스트 셀을 실행해 `테스트 통과!`가 나오는지 확인하세요.

### 실습 1 Singleton: `Logger`

- `Logger()`를 몇 번 호출해도 **같은 객체**를 반환
- `log(msg)` → 내부 리스트에 메시지 추가
- `logs` 속성 → 지금까지의 메시지 리스트

In [ ]:
class Logger:
    _instance = None

    def __new__(cls):
        pass  # TODO

    def log(self, msg):
        pass  # TODO

In [ ]:
# 테스트: 그대로 실행하세요
Logger._instance = None          # 테스트를 다시 실행해도 같은 결과가 나오도록 초기화
a = Logger()
b = Logger()
a.log("서버 시작")
b.log("접속 1건")
print(Logger().logs)
assert a is b
assert Logger().logs == ["서버 시작", "접속 1건"]
print("테스트 통과!")

### 실습 2 Observer: 유튜브 채널 구독

- `Channel(name)`: `subscribe(sub)`, `unsubscribe(sub)`, `upload(title)`
- `upload()`하면 모든 구독자의 `notify(channel_name, title)` 호출
- `Subscriber(name)`: `notify()`가 호출되면 `inbox` 리스트에 `"[채널명] 제목"` 추가

In [ ]:
class Subscriber:
    def __init__(self, name):
        self.name = name
        self.inbox = []

    def notify(self, channel_name, title):
        pass  # TODO

class Channel:
    def __init__(self, name):
        self.name = name
        # TODO

    def subscribe(self, sub):
        pass  # TODO

    def unsubscribe(self, sub):
        pass  # TODO

    def upload(self, title):
        pass  # TODO

In [ ]:
# 테스트: 그대로 실행하세요
ch = Channel("파이썬TV")
kim, lee = Subscriber("kim"), Subscriber("lee")
ch.subscribe(kim)
ch.subscribe(lee)
ch.upload("클래스 기초")
ch.unsubscribe(lee)
ch.upload("디자인 패턴")
print(kim.inbox)
print(lee.inbox)
assert kim.inbox == ["[파이썬TV] 클래스 기초", "[파이썬TV] 디자인 패턴"]
assert lee.inbox == ["[파이썬TV] 클래스 기초"]
print("테스트 통과!")

### 실습 3 Strategy: 배송비 계산 (함수 버전)

전략을 **함수**로 만들어 전달하세요.

- `standard(weight)` → 3,000원 고정
- `express(weight)` → 5,000원 + kg당 1,000원
- `free(weight)` → 0원
- `shipping_cost(weight, strategy=standard)` → 전략 함수를 호출한 결과 반환

In [ ]:
def standard(weight):
    pass  # TODO

def express(weight):
    pass  # TODO

def free(weight):
    pass  # TODO

def shipping_cost(weight, strategy=standard):
    pass  # TODO

In [ ]:
# 테스트: 그대로 실행하세요
for s in (standard, express, free):
    print(s.__name__, shipping_cost(3, s))
assert shipping_cost(3) == 3000
assert shipping_cost(3, express) == 8000
assert shipping_cost(10, free) == 0
print("테스트 통과!")

### 실습 4 Factory: 도형 공장

- `Circle(r)`, `Rectangle(w, h)`는 `area()` 메서드를 가짐 (원주율은 `math.pi`)
- `create_shape(kind, *args)` → 딕셔너리에서 클래스를 찾아 객체 생성
- 등록되지 않은 `kind`면 `ValueError`

In [ ]:
import math

class Circle:
    pass  # TODO

class Rectangle:
    pass  # TODO

SHAPES = {}  # TODO: 문자열 → 클래스

def create_shape(kind, *args):
    pass  # TODO

In [ ]:
# 테스트: 그대로 실행하세요
c = create_shape("circle", 1)
r = create_shape("rectangle", 3, 4)
print(round(c.area(), 2), r.area())
assert round(c.area(), 2) == 3.14
assert r.area() == 12
try:
    create_shape("star")
    raise AssertionError("ValueError가 발생해야 합니다")
except ValueError:
    pass
print("테스트 통과!")

## 7. 확인 문제

**Q1.** 다음 각 상황에 가장 알맞은 패턴은?

1. 로그를 남기는 객체가 프로그램 전체에 하나만 있어야 한다.
2. 주식 가격이 바뀔 때마다 여러 화면(차트, 알림, 기록)을 갱신해야 한다.
3. 사용자가 선택한 정렬 방식(이름순, 가격순, 인기순)에 따라 상품을 정렬한다.
4. 외부 지도 라이브러리의 `getLatLng()`를 우리 코드의 `location()` 형식에 맞춰 써야 한다.
5. 설정 파일에 적힌 `"csv"`, `"json"` 값에 따라 알맞은 파서 객체를 만들어야 한다.

**Q2.** 싱글턴 예제에서 `c1 = AppConfig()`, `c2 = AppConfig()`를 실행하면 "AppConfig 객체 생성"은 몇 번 출력되는가?

**Q3.** 데코레이터 예제에서 `Syrup(Whip(Americano())).cost()`의 값은?

**Q4.** 빌더 패턴의 설정 메서드가 `return self`를 하지 않으면 어떤 문제가 생기는가?